# Pre-Requisite Assistant RAG LCEL


In [ ]:
!pip install langsmith langchain langchain_community langchain_groq langchain_core pydantic chromadb

In [ ]:
import os
os.environ['LANGSMITH_PROJECT'] = "langgraph-prerequisite"

In [ ]:
from google.colab import userdata
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
os.environ['LANGSMITH_API_KEY'] = userdata.get('LANGCHAIN_API_KEY')
os.environ['LANGSMITH_ENDPOINT']= "https://api.smith.langchain.com"
os.environ['LANGSMITH_TRACING'] = "true"

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma

In [ ]:
from langchain_groq import ChatGroq
llm=ChatGroq(model_name="llama-3.1-8b-instant")

In [ ]:
while True:
  question=input("type your question. if you want to quit the chat write quit")
  if question !="quit":
    print(llm.invoke(question).content)
  else:
    print("goodbye take care yourself")
    break


type your question. if you want to quit the chat write quithello
Hello. Is there something I can help you with or would you like to chat?
type your question. if you want to quit the chat write quitquit
goodbye take care yourself


In [ ]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import AIMessage

In [ ]:
store={}

In [ ]:
def get_session_history(session_id: str) -> BaseChatMessageHistory:
  if session_id not in store:
    store[session_id] = InMemoryChatMessageHistory()
  return store[session_id]

In [ ]:
config = {"configurable": {"session_id": "firstchat"}}

In [ ]:
model_with_memory = RunnableWithMessageHistory(llm, get_session_history)

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [ ]:
model_with_memory.invoke(("Hi! I'm Yousuf Paryani"),config=config).content

"Hello Yousuf Paryani, how are you today? It's nice to meet you. Is there something I can help you with or would you like to chat?"

In [ ]:
model_with_memory.invoke(("tell me what is my name?"),config=config).content

'Your name is Yousuf Paryani.'

In [ ]:
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
### Reading the txt files from source directory

loader = DirectoryLoader('data', glob="./*.txt", loader_cls=TextLoader)
docs = loader.load()

In [ ]:
docs

[Document(metadata={'source': 'data/japan.txt'}, page_content='Industrial revival hope for Japan\n\nJapanese industry is growing faster than expected, boosting hopes that the country\'s retreat back into recession is over.\n\nIndustrial output rose 2.1% - adjusted for the time of year - in January from a month earlier. At the same time, retail sales picked up faster than at any time since 1997. The news sent Tokyo shares to an eight-month high, as investors hoped for a recovery from the three quarters of contraction seen from April 2004 on. The Nikkei 225 index ended the day up 0.7% at 11,740.60 points, with the yen strengthening 0.7% against the dollar to 104.53 yen. Weaker exports, normally the engine for Japan\'s economy in the face of weak domestic demand, had helped trigger a 0.1% contraction in the final three months of last year after two previous quarters of shrinking GDP. Only an exceptionally strong performance in the early months of 2004 kept the year as a whole from showing

In [ ]:
### Creating Chunks using RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=10,
    length_function=len,

)

new_docs = text_splitter.split_documents(documents=docs)
doc_strings = [doc.page_content for doc in new_docs]

In [ ]:
###  BGE Embddings

from langchain_community.embeddings import HuggingFaceBgeEmbeddings

model_name = "BAAI/bge-base-en-v1.5"
model_kwargs = {'device': 'cuda'}
encode_kwargs = {'normalize_embeddings': True} # set True to compute cosine similarity
embeddings = HuggingFaceBgeEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)

db = Chroma.from_documents(new_docs, embeddings)
retriever = db.as_retriever(search_kwargs={"k": 4})

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = PromptTemplate.from_template(template)

In [ ]:
prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\n{context}\n\nQuestion: {question}\n')

In [ ]:
retrieval_chain = (
    RunnableParallel({"context": retriever, "question": RunnablePassthrough()})
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
question ="what is llama3? can you highlight 3 important points?"
print(retrieval_chain.invoke(question))

Based on the provided context, here's what can be inferred about Llama 3:

Llama 3 appears to be a version of the Llama model, likely a conversational AI model. Here are 3 important points:

1. **Release Date**: Although the exact release date is not specified, there is a mention of "April 2024" as a possible release month.
2. **Parameter Version**: The 8B parameter version of Llama 3 is described in a document, implying that Llama 3 comes in different parameter versions, with 8B being one of them.
3. **Integration and Services**: Llama 3 is used in various services, including a website and other services (whose names are not specified). This suggests that Llama 3 is being utilized in multiple applications and platforms.


## Let's Start with Tools and Agents

In [ ]:
!pip install wikipedia

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=586478bf7c42ddf49bb06aa9c7fc0bd7172018c13ce7841268c7a7f947e03b98
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia


In [ ]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [ ]:
api_wrapper = WikipediaAPIWrapper(top_k_results=5, doc_content_chars_max=100)

In [ ]:
tool = WikipediaQueryRun(api_wrapper=api_wrapper)

In [ ]:
tool.name

'wikipedia'

In [ ]:
tool.description

'A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.'

In [ ]:
tool.args

{'query': {'description': 'query to look up on wikipedia',
  'title': 'Query',
  'type': 'string'}}

In [ ]:
tool.return_direct

False

In [ ]:
print(tool.run("langchain"))

Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of 


In [ ]:
tool.run("langchain")

'Page: LangChain\nSummary: LangChain is a software framework that helps facilitate the integration of '

In [ ]:
from pydantic import BaseModel, Field

In [ ]:
class WikiInputs(BaseModel):
  query: str = Field(description="query to look up in Wikipedia, should be 3 or less words")

In [ ]:
tool = WikipediaQueryRun(
    name="wiki-tool",
    description="look up things in wikipedia",
    args_schema=WikiInputs,
    api_wrapper=api_wrapper,
    return_direct=True,
)

In [ ]:
tool.name

'wiki-tool'

In [ ]:
tool.description

'look up things in wikipedia'

In [ ]:
tool.args

{'query': {'description': 'query to look up in Wikipedia, should be 3 or less words',
  'title': 'Query',
  'type': 'string'}}

In [ ]:
tool.return_direct

True

In [ ]:
print(tool.run({"query": "langchain"}))

Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of 


## youtube search tool

In [ ]:
from langchain_community.tools import YouTubeSearchTool

In [ ]:
tool=YouTubeSearchTool()

In [ ]:
tool.name

'youtube_search'

In [ ]:
tool.description

'search for youtube videos associated with a person. the input to this tool should be a comma separated list, the first part contains a person name and the second a number that is the maximum number of video results to return aka num_results. the second part is optional'

In [ ]:
!pip install youtube_search
from youtube_search import YoutubeSearch

tool.run("sunny savita")

"['https://www.youtube.com/watch?v=Tf2ZzrCBJUI&pp=ygUMc3Vubnkgc2F2aXRh', 'https://www.youtube.com/watch?v=xv-frxET5Mo&pp=ygUMc3Vubnkgc2F2aXRh']"

In [ ]:
!pip install langchain-tavily


In [ ]:
os.environ["TAVILY_API_KEY"]= userdata.get("TAVILY_API_KEY")

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults

In [ ]:
tool = TavilySearchResults()

/tmp/ipykernel_17600/1442711075.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tool = TavilySearchResults()


In [ ]:
tool.invoke({"query": "what is new news about AI Transformers?"})

[{'title': 'Transformers Revolutionized AI. What Will Replace Them?',
  'url': 'https://www.forbes.com/sites/robtoews/2023/09/03/transformers-revolutionized-ai-what-will-replace-them/',
  'content': 'Transformers’ influence reaches well beyond text and images. The most advanced robotics research today relies on transformers. Indeed, Google’s most recent robotics work is actually named RT-2, where the T stands for “transformer.” Similarly, one of the most promising new avenues of research in the field of autonomous vehicles is the use of vision transformers. Transformer-based models have unlocked breathtaking new possibilities in biology, including the ability to design customized proteins and nucleic acids that have never before existed in nature. [...] Transformers have become the foundation of modern artificial intelligence. Virtually every advanced AI system is based on transformers; every AI researcher is accustomed to working with them. Transformers have been optimized by thousand